# COSMOS morphology linear probe — jwst_dino teacher

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import warnings; warnings.simplefilter('ignore')

sys.path.insert(0, os.path.join('..', '..'))          # jwst_dino/ (model + data import)
from model.jwst_dino import load_teacher_backbone
from dataset import CosmosMorphDataset, CLASS_NAMES

ROOT   = '~/ssl_outthere/data/image'
CKPT   = '/home/yacheng/ssl_outthere/encoder_image/jwst_dino/outputs/jwst_dino_ps6_st3/version_6/checkpoints/last.ckpt'
MORPH  = '../../../../data/survey/cosmos_2025/COSMOSWeb_mastercatalog_v1_ml_morph.fits'

DELTA_THRESHOLD   = 1     # keep delta_f150w < this (classification confidence)
EXCLUDE_IRREGULAR = False    # drop the noisy irregular class
BALANCED          = True    # undersample to the minority class count
TEST_FRAC         = 0.3
SEED              = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
# 1) frozen teacher backbone (crop_size comes from the checkpoint's training config)
net  = load_teacher_backbone(CKPT, DEVICE)
crop = net.crop_size
print('crop_size:', crop)

In [ ]:
# 2) COSMOS f150w cutouts + ml_morph labels (cross-matched by id, no-irr, balanced)
ds = CosmosMorphDataset(
    root=ROOT, morph_catalog=MORPH, filter='f150w', crop_size=crop,
    delta_threshold=DELTA_THRESHOLD, exclude_irregular=EXCLUDE_IRREGULAR,
    balanced=BALANCED, seed=SEED,
)
loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

In [ ]:
# 3) extract CLS embeddings
@torch.no_grad()
def extract(net, loader, device):
    embs, labels = [], []
    for imgs, ys in tqdm(loader, desc='embeddings'):
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16,
                            enabled=device.type == 'cuda'):
            cls = net(imgs.to(device))['cls']
        embs.append(cls.float().cpu().numpy()); labels.append(np.asarray(ys))
    return np.concatenate(embs), np.concatenate(labels)

X, y = extract(net, loader, DEVICE)
print('embeddings:', X.shape, ' labels:', y.shape)

In [ ]:
# 4) linear probe: standardize -> logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_FRAC, random_state=SEED, stratify=y)
scaler = StandardScaler().fit(Xtr)
clf = LogisticRegression(max_iter=2000, C=1.0).fit(scaler.transform(Xtr), ytr)
pred = clf.predict(scaler.transform(Xte))

names = [CLASS_NAMES[c] for c in sorted(np.unique(y))]
print(f'=== linear probe ({len(Xtr)} train / {len(Xte)} test) ===')
print(f'accuracy : {accuracy_score(yte, pred):.4f}')
print(f'macro-F1 : {f1_score(yte, pred, average="macro"):.4f}')
print(classification_report(yte, pred, target_names=names, digits=3))

In [ ]:
# 5) confusion matrix (row-normalized)
cm = confusion_matrix(yte, pred)
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center',
                color='white' if cmn[i, j] > 0.5 else 'black')
ax.set_title(f'COSMOS morphology probe (include irr {EXCLUDE_IRREGULAR})')
fig.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout(); plt.show()